# plot_sarima.py
Plots train/test/forecast lines for six ZIP codes in one figure.

In [1]:
"""Plot train/test/forecast lines for all six ZIP codes in one figure.

Input: outputs/sarima_data.csv, outputs/sarima_forecasts.csv
Output: outputs/plots/sarima_forecast_all_zips.png
"""
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
DATA_FILE = PROJECT_ROOT / "outputs" / "sarima_data.csv"
FORECAST_FILE = PROJECT_ROOT / "outputs" / "sarima_forecasts.csv"
PLOT_FILE = PROJECT_ROOT / "outputs" / "plots" / "sarima_forecast_all_zips.png"
ZIPS = {
    "02126": "Mattapan",
    "02116": "Back Bay",
    "02150": "Chelsea",
    "02139": "Cambridge",
    "01840": "Lawrence",
    "02481": "Wellesley",
}
TRAIN_END = "2022-12-31"

In [2]:
def money_fmt(x, _):
    return f"${x/1e6:.1f}M" if x >= 1e6 else f"${x/1e3:.0f}K"

In [3]:
def main():
    if not FORECAST_FILE.exists():
        raise FileNotFoundError(
            "Missing outputs/sarima_forecasts.csv. Run scripts/modeling/train_sarima.py first."
        )

    data = pd.read_csv(DATA_FILE, dtype={"zip_code": str}, parse_dates=["date"])
    forecasts = pd.read_csv(FORECAST_FILE, dtype={"zip_code": str}, parse_dates=["date"])
    PLOT_FILE.parent.mkdir(parents=True, exist_ok=True)

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle("SARIMAX Forecast - Actual vs Predicted", fontweight="bold", fontsize=14)

    for ax, zip_code in zip(axes.ravel(), ZIPS):
        sub = data[data["zip_code"] == zip_code].sort_values("date").set_index("date")
        fsub = forecasts[forecasts["zip_code"] == zip_code].sort_values("date").set_index("date")

        train = sub.loc[sub.index <= TRAIN_END, "zhvi"]
        test = fsub["actual_zhvi"].astype(float)
        pred = fsub["pred_zhvi"].astype(float)
        test_arr = test.to_numpy(dtype=float)
        pred_arr = pred.to_numpy(dtype=float)
        mape = np.mean(np.abs((test_arr - pred_arr) / test_arr)) * 100

        ax.plot(train.index, train, color="steelblue", lw=1.8, label="Train")
        ax.plot(test.index, test, color="black", lw=2, label="Actual")
        ax.plot(pred.index, pred, color="tomato", lw=2, ls="--", label="Forecast")
        ax.axvspan(test.index[0], test.index[-1], color="tomato", alpha=0.05)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(money_fmt))
        ax.set_title(f"{zip_code} - {ZIPS[zip_code]}  |  MAPE {mape:.1f}%", fontsize=11)
        ax.legend(frameon=False, fontsize=8)
        ax.grid(axis="y", alpha=0.3)
        for spine in ax.spines.values():
            spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(PLOT_FILE, dpi=180, bbox_inches="tight")
    plt.close()

In [4]:
if __name__ == "__main__":
    main()